# Step 5 (Phase 3) — LoRA + BitFit + Full-FT + Linear-Probe on CIFAR-FS
### Kaggle version

Git-based runner. Clones the repo into `/kaggle/working` (the writable, persistent-for-this-session disk), builds the frozen Bertinetto split, and runs all **8** configs (4 adapters × 2 heads) end-to-end, producing `results/phase3_*_metrics.json`.

**PREREQUISITE:** the Step 5 code must be pushed to the branch you set in `BRANCH` below (default `main`). By repo convention a human commits/pushes — do that first, then run this notebook top-to-bottom.

**Before running:** in the notebook **Settings** panel (right sidebar), turn **Internet: On** and **Accelerator: GPU** (T4 x2 or P100). Then use **Add Data** to attach the Kaggle Dataset that contains `cifar-100-python.tar.gz` (see the setup guide for how to create it).

Runtime: LoRA / BitFit / Linear-Probe are close to Bottleneck's cost; the two **Full-FT** runs are the slow ones (backprop through the whole backbone) and may take ~20-40 min each on a T4. The run loop continues on error, so one failed config never blocks the others.

## 0. GPU + environment check

In [1]:
import torch, sys
print('python :', sys.version.split()[0])
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable a GPU: notebook Settings (right sidebar) > Accelerator > GPU T4 x2 or P100'

python : 3.12.13
torch  : 2.10.0+cu128
cuda   : True | Tesla T4


## 1. Clone the repo + install deps

Set `BRANCH` to whatever branch has the Step 5 commit. Re-running the cell pulls the latest instead of re-cloning. Requires **Internet: On** in the notebook Settings panel.

In [2]:
import os, subprocess

REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'          # <-- set to the branch that has your Step 5 push
REPO_DIR = '/kaggle/working/thesis'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
!pip -q install -r requirements.txt

# Sanity: the Step 5 adapters must be present in this checkout.
import importlib.util
for f in ['src/adapters/lora.py', 'src/adapters/bitfit.py', 'src/adapters/full_ft.py',
          'src/adapters/linear_probe.py']:
    assert os.path.exists(f), f'MISSING {f} — did you push the Step 5 commit to {BRANCH}?'
print('Step 5 adapters present — good.')

Cloning into '/kaggle/working/thesis'...


cwd: /kaggle/working/thesis
f364ad6 Merge pull request #3 from notAvailable73/step4-best-of-both
Step 5 adapters present — good.


## 2. Build the frozen CIFAR-FS Bertinetto split

`data/` is gitignored, so materialize the canonical 64/16/20 split once (downloads CIFAR-100 to local disk). Do NOT hand-edit the JSON it writes.

In [3]:
!python scripts/build_cifar_fs_split.py
import json
sp = json.load(open('data/cifar_fs_split.json'))
print('split status:', sp.get('_status'))
print('sizes:', {k: len(v) for k, v in sp.items() if isinstance(v, list)})

wrote /kaggle/working/thesis/data/cifar_fs_split.json  (64/16/20, disjoint, union=100, status=canonical_bertinetto_via_torchmeta)
split status: canonical_bertinetto_via_torchmeta
sizes: {'train': 64, 'val': 16, 'test': 20}


## 2b. Stage CIFAR-100 from your attached Kaggle Dataset (avoids the slow cs.toronto.edu download)

`cs.toronto.edu` serves `cifar-100-python.tar.gz` very slowly and unreliably from cloud notebooks — a full run can stall for a long time and then fail. This cell copies a pre-staged, md5-verified tarball from a Kaggle Dataset you attached (via **Add Data**) into `data/`; torchvision and `ensure_archive` both short-circuit when the file is already present, so **nothing downstream — the tests OR the 8 training runs — ever touches cs.toronto.edu.**

**ONE-TIME SETUP:** create a Kaggle Dataset (from any machine that already has the file, e.g. your local repo's `data/` directory) containing `cifar-100-python.tar.gz` (md5 `eb9058c3a382ffc7106e4002c42a8d85`), then attach it to this notebook with **Add Data**. Set `KAGGLE_DATASET_DIR` below to match the path Kaggle mounts it at (shown under **Add Data** or in the file browser on the left, usually `/kaggle/input/<your-dataset-slug>`).

In [4]:
import os, shutil

# <-- matches what you see in the left-hand Data pane
KAGGLE_DATASET_DIR = '/kaggle/input/datasets/notavailable73/cifar100/cifar-100-python'
dst = 'data/cifar-100-python'

os.makedirs('data', exist_ok=True)
if os.path.isdir(dst) and os.path.exists(os.path.join(dst, 'train')):
    print(f'[cifar] already present at {dst} - nothing to do')
else:
    if not os.path.isdir(KAGGLE_DATASET_DIR):
        raise FileNotFoundError(
            'cifar-100-python folder not found in the attached Kaggle dataset.\n'
            f'Looked in: {KAGGLE_DATASET_DIR}\n'
            'Check the left-hand Data pane for the exact path and update '
            'KAGGLE_DATASET_DIR above.')
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(KAGGLE_DATASET_DIR, dst)
    print(f'[cifar] copied {dst} from Kaggle input '
          f'({sum(os.path.getsize(os.path.join(dst, f)) for f in os.listdir(dst))/1e6:.0f} MB)')

[cifar] copied data/cifar-100-python from Kaggle input (186 MB)


## 3. (optional) Run the test suite

Confirms the new adapters + the rest of the suite pass before burning GPU on the runs. Expect the Step-4/4.5 tests plus the four new Step-5 files.

In [5]:
# Pre-fetch the frozen backbone's ImageNet weights (~45 MB) with a VISIBLE
# progress bar. The test suite builds the real ResNet-18, which otherwise
# downloads these weights silently on first use — on a slow link that looks
# exactly like a hang. Requires Internet: On in notebook Settings.
from torchvision.models import resnet18, ResNet18_Weights
_ = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
print('resnet18 ImageNet weights cached.')

# Run pytest UNBUFFERED and streamed (no `| tail`, which hides all output
# until the run finishes). `-v` prints one line per test so you can see it
# is alive and which test — if any — is actually slow.
!python -u -m pytest -v --durations=10

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 217MB/s]


resnet18 ImageNet weights cached.
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/thesis
plugins: anyio-4.13.0, typeguard-4.5.1, langsmith-0.7.34
collected 91 items                                                             

tests/test_adapters.py::test_bottleneck_is_identity_at_init PASSED       [  1%]
tests/test_adapters.py::test_build_adapter_factory_bottleneck PASSED     [  2%]
tests/test_adapters.py::test_build_adapter_lora_requires_backbone PASSED [  3%]
tests/test_adapters.py::test_bottleneck_param_count PASSED               [  4%]
tests/test_bitfit.py::test_bitfit_unfreezes_only_biases PASSED           [  5%]
tests/test_bitfit.py::test_bitfit_param_count_matches_analytical PASSED  [  6%]
tests/test_bitfit.py::test_bitfit_backbone_trainable_flag PASSED         [  7%]
tests/test_bitfit.py::test_bitfit_via_factory_requ

## 4. Run all 8 configs (train + 600-episode eval)

One JSON per config in `results/phase3_*_metrics.json`. Already-finished configs are skipped, so you can re-run this cell to resume after an interruption (within the same session — see the note in Section 6 about Kaggle session limits). `USE_TINYIMAGENET=True` adds the near-OOD pool (downloads a ~240 MB zip once); set it to `False` if that download is flaky.

In [6]:
import subprocess, os

USE_TINYIMAGENET = True
NUM_EPISODES     = 600
SUFFIX           = 'phase3'

# (config-basename, adapter.type, interpretation)
RUNS = [
    ('exp_phase3_lora_evidential',        'lora',         'evidential'),
    ('exp_phase3_lora_softmax',           'lora',         'softmax'),
    ('exp_phase3_bitfit_evidential',      'bitfit',       'evidential'),
    ('exp_phase3_bitfit_softmax',         'bitfit',       'softmax'),
    ('exp_phase3_full_ft_evidential',     'full_ft',      'evidential'),
    ('exp_phase3_full_ft_softmax',        'full_ft',      'softmax'),
    ('exp_phase3_linear_probe_evidential','linear_probe', 'evidential'),
    ('exp_phase3_linear_probe_softmax',   'linear_probe', 'softmax'),
]

def result_path(adapter, interp):
    return f'results/{SUFFIX}_{adapter}_prototype-{interp}_metrics.json'

def run(cmd):
    # Inherit stdout/stderr so the training log streams live into the cell
    # (do NOT capture — captured subprocess output can silently vanish).
    print('>>>', ' '.join(cmd), flush=True)
    return subprocess.run(cmd).returncode

status = {}
for name, adapter, interp in RUNS:
    out = result_path(adapter, interp)
    if os.path.exists(out):
        print(f'== SKIP {name} (found {out}) ==', flush=True)
        status[name] = 'skip (exists)'
        continue
    print(f'\n{"="*72}\n== {name} ==\n{"="*72}', flush=True)
    cfg = f'configs/{name}.yaml'
    try:
        rc = run(['python', 'scripts/train.py', '--config', cfg, '--wandb-mode', 'disabled'])
        if rc != 0:
            status[name] = f'TRAIN failed (rc={rc})'; continue
        eval_cmd = ['python', 'scripts/evaluate.py', '--config', cfg,
                    '--num-episodes', str(NUM_EPISODES), '--wandb-mode', 'disabled',
                    '--results-suffix', SUFFIX]
        if USE_TINYIMAGENET:
            eval_cmd.append('--use-tinyimagenet')
        rc = run(eval_cmd)
        status[name] = 'OK' if (rc == 0 and os.path.exists(out)) else f'EVAL failed (rc={rc})'
    except Exception as e:
        status[name] = f'EXCEPTION: {e}'

print('\n' + '=' * 40 + '\nRUN STATUS\n' + '=' * 40)
for name, _, _ in RUNS:
    print(f'  {name:38s} {status.get(name, "not run")}')


== exp_phase3_lora_evidential ==
>>> python scripts/train.py --config configs/exp_phase3_lora_evidential.yaml --wandb-mode disabled
[14:41:37] INFO bpeft.train: config: configs/exp_phase3_lora_evidential.yaml  seed: 42  trainer.type: episodic
[14:41:37] INFO bpeft.train: wandb: disabled (no-op logger)
[14:41:40] INFO bpeft.train: trainable params: 12,290
[14:42:30] INFO bpeft.train: epoch   1/30  train_loss=0.5132  train_acc=0.796  val_loss=0.6715  val_acc=0.741  kl_w=0.010  mean_ev=2.9307  grad_norm=0.4594  global_step=100
[14:43:17] INFO bpeft.train: epoch   2/30  train_loss=0.4727  train_acc=0.803  val_loss=0.6706  val_acc=0.728  kl_w=0.020  mean_ev=2.2042  grad_norm=0.4215  global_step=200
[14:44:05] INFO bpeft.train: epoch   3/30  train_loss=0.4414  train_acc=0.821  val_loss=0.6369  val_acc=0.737  kl_w=0.030  mean_ev=2.0152  grad_norm=0.4403  global_step=300
[14:44:54] INFO bpeft.train: epoch   4/30  train_loss=0.4283  train_acc=0.831  val_loss=0.6141  val_acc=0.764  kl_w=0.040  

## 5. Summary table

Reads every `results/phase3_*_metrics.json` and shows the headline numbers, alongside the Step 4.5 Bottleneck baseline of record.

In [7]:
import glob, json
import pandas as pd

rows = []
for f in sorted(glob.glob('results/phase3_*_metrics.json')) + \
         sorted(glob.glob('results/step45_*_metrics.json')):
    d = json.load(open(f))
    rows.append({
        'file'      : os.path.basename(f),
        'adapter'   : d.get('adapter_type'),
        'interp'    : d.get('interpretation'),
        'n_params'  : d.get('n_params'),
        'accuracy'  : round(d.get('accuracy_mean', float('nan')), 4),
        'f1_macro'  : round(d.get('f1_macro_mean', float('nan')), 4),
        'ece'       : round(d.get('ece_pooled', float('nan')), 4),
        'ece_ts'    : round(d['ece_ts'], 4) if 'ece_ts' in d else None,
        'brier'     : round(d.get('brier_mean', float('nan')), 4),
        'auroc(prim)': round(d.get('ood_auroc_mean', float('nan')), 4),
        'best_val_ep': d.get('best_val_epoch'),
    })
df = pd.DataFrame(rows)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
df

,file,adapter,interp,n_params,accuracy,f1_macro,ece,ece_ts,brier,auroc(prim),best_val_ep
0,phase3_bitfit_prototype-evidential_metrics.json,bitfit,evidential,4802,0.8690,0.8674,0.3328,NaN,0.3418,0.9261,2
1,phase3_bitfit_prototype-softmax_metrics.json,bitfit,softmax,4800,0.8807,0.8791,0.1465,0.0055,0.2110,0.8539,1
2,phase3_full_ft_prototype-evidential_metrics.json,full_ft,evidential,11176514,0.8882,0.8856,0.3383,NaN,0.3220,0.9090,6
3,phase3_full_ft_prototype-softmax_metrics.json,full_ft,softmax,11176512,0.9047,0.9027,0.0728,0.0135,0.1536,0.8938,6
4,phase3_linear_probe_prototype-evidential_metri...,linear_probe,evidential,2,0.8741,0.8728,0.6234,NaN,0.7078,0.7079,1
5,phase3_linear_probe_prototype-softmax_metrics....,linear_probe,softmax,0,0.8741,0.8728,0.4476,0.0297,0.4554,0.8551,0
6,phase3_lora_prototype-evidential_metrics.json,lora,evidential,12290,0.8278,0.8241,0.3253,NaN,0.3915,0.8946,4
7,phase3_lora_prototype-softmax_metrics.json,lora,softmax,12288,0.8601,0.8571,0.0873,0.0176,0.2149,0.7829,6
8,step45_bottleneck_prototype-evidential_metrics...,bottleneck,evidential,16914,0.8835,0.8819,0.2846,NaN,0.2849,0.9141,8
9,step45_bottleneck_prototype-softmax_metrics.json,bottleneck,softmax,16912,0.8747,0.8729,0.0817,0.0406,0.1905,0.8378,2


## 6. Results persistence

Unlike Colab, there's no Drive mount needed here: anything left under `/kaggle/working` (which includes `results/`, since the repo was cloned into `/kaggle/working/thesis`) is automatically kept when you click **Save Version** (commit) on this notebook — it becomes downloadable notebook output. This cell just zips the 8 JSONs for a single convenient download.

**Note on session limits:** Kaggle GPU sessions have a runtime cap (check current limits under Settings). If you expect the 8 runs to take longer than that, re-run this notebook in stages — Section 4 automatically skips configs whose result JSON already exists in `results/`, so committing and re-opening the notebook (with the same attached dataset) resumes cleanly **as long as `/kaggle/working` wasn't reset** between sessions. For a hard guarantee across session resets, add the previous commit's output as an input dataset and copy its JSONs into `results/` before Section 4.

In [8]:
import shutil, glob, os

files = glob.glob('results/phase3_*')
print(f'{len(files)} result files ready in {os.path.abspath("results")}')

# Zip everything into /kaggle/working for a single-file download from the
# notebook's Output pane once you commit.
shutil.make_archive('/kaggle/working/phase3_results', 'zip', 'results')
print('zipped to /kaggle/working/phase3_results.zip')

32 result files ready in /kaggle/working/thesis/results
zipped to /kaggle/working/phase3_results.zip
